# ComplianceGPT Pipeline — Unified Colab Entrypoints

This notebook is designed for **Google Colab ** execution and produces a **verifier-ready Final Answer Contract**.

**Single source of truth:** `src/compliancegpt/generator/citation_contract_80053.md`


## Run Batch (Gold Sets)

This notebook runs the pipeline over **Rev4 (36Q)** or **Rev5 (100Q)** gold set and saves outputs to Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, sys, json
PROJECT_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"
assert os.path.isdir(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import compliancegpt
print("compliancegpt.__file__ =", compliancegpt.__file__)


compliancegpt.__file__ = None


In [3]:
# --- Canonical paths (Drive) ---
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
PIPELINE_DIR = os.path.join(PROJECT_ROOT, "src", "compliancegpt", "pipeline")

REV5_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev5_catalog.jsonl")
REV4_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev4_catalog.jsonl")

REV5_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev5_gold-set_100q.csv")
REV4_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev4_gold-set_36q.csv")

QUR_REV5_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev5.csv")
QUR_REV4_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev4.csv")
QUR_ERROR_BANK_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_error_bank.csv")

ERROR_BANK_PATH = os.path.join(DATA_DIR, "error_bank", "error_bank_v1.csv")

ORG_PROFILE_REV5 = os.path.join(DATA_DIR, "ODP", "rev5", "org_profile_example_rev5.yaml")
ORG_PROFILE_REV4 = os.path.join(DATA_DIR, "ODP", "rev4", "org_profile_example_rev4.yaml")

for p in [REV5_CATALOG_PATH, REV4_CATALOG_PATH, REV5_GOLD_PATH, REV4_GOLD_PATH]:
    if not os.path.exists(p):
        print("[WARN] Missing:", p)


In [4]:
# --- Helpers: load gold sets and QUR rewrite maps ---
from typing import Any, Dict, List, Optional
import pandas as pd

def load_gold(framework_version: str) -> pd.DataFrame:
    path = REV5_GOLD_PATH if framework_version.lower() == "rev5" else REV4_GOLD_PATH
    # The gold CSVs are often not UTF-8; use latin1 for robustness.
    df = pd.read_csv(path, encoding="latin1")
    # Standardize columns
    if "id" in df.columns and "query_id" not in df.columns:
        df["query_id"] = df["id"].astype(str)
    if "question" in df.columns and "original_query" not in df.columns:
        df["original_query"] = df["question"].astype(str)
    return df

def load_qur_map(framework_version: str) -> Dict[str, List[str]]:
    path = QUR_REV5_PATH if framework_version.lower() == "rev5" else QUR_REV4_PATH
    if not os.path.exists(path):
        print(f"[INFO] QUR CSV not found: {path}")
        return {}
    df = pd.read_csv(path)
    # Expected columns from QUR generator: query_id, rewrite_rank, rewritten_query
    required = {"query_id", "rewrite_rank", "rewritten_query"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"Unexpected QUR CSV schema. Missing {required - set(df.columns)}. Columns={list(df.columns)}")
    out: Dict[str, List[str]] = {}
    for qid, g in df.groupby(df["query_id"].astype(str)):
        g2 = g.sort_values("rewrite_rank")
        out[qid] = [str(x) for x in g2["rewritten_query"].tolist() if isinstance(x, str) and x.strip()]
    return out

def get_gold_row(df: pd.DataFrame, qid: Any) -> Dict[str, Any]:
    qid_str = str(qid)
    m = df[df["query_id"].astype(str) == qid_str]
    if m.empty:
        raise KeyError(f"query_id not found in gold set: {qid_str}")
    # Use the first match (ids should be unique)
    return dict(m.iloc[0].to_dict())


In [5]:
# --- Build pipeline (single place to configure knobs) ---
from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline

def build_pipeline(
    *,
    framework_version: str,
    use_qur_live: bool,
    use_org_profile: bool,
    verify_strict_extras: bool = False,
    verify_strict_verbatim: bool = True,
    verify_strict_version: bool = False,
):
    fw = framework_version.lower().strip()
    if fw not in {"rev4", "rev5"}:
        raise ValueError(f"framework_version must be rev4 or rev5 (got {framework_version!r})")

    ccs_path = REV5_CATALOG_PATH if fw == "rev5" else REV4_CATALOG_PATH
    org_profile = (ORG_PROFILE_REV5 if fw == "rev5" else ORG_PROFILE_REV4) if use_org_profile else None
    if org_profile and not os.path.exists(org_profile):
        print(f"[WARN] org_profile_path not found; continuing without profile: {org_profile}")
        org_profile = None

    pipe = ComplianceGPTPipeline(
        framework_version=fw,
        ccs_path=ccs_path,
        use_qur=bool(use_qur_live),
        org_profile_path=org_profile,
        verify_strict_extras=bool(verify_strict_extras),
        verify_strict_verbatim=bool(verify_strict_verbatim),
        verify_strict_version=bool(verify_strict_version),
    )
    return pipe


[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib


### Configure batch run

In [6]:
# === USER CONFIG ===
FRAMEWORK_VERSION = "rev5"   # "rev4" or "rev5"

# Rewrites
USE_QUR_PRECOMPUTED = True   # use data/qur_outputs/*.csv if available
USE_QUR_LIVE = False         # if True, the pipeline can generate rewrites if none are provided (slower)

# Org profile (only needed for FILL_FROM_PROFILE)
USE_ORG_PROFILE = False

# Verification strictness knobs
VERIFY_STRICT_EXTRAS = False
VERIFY_STRICT_VERBATIM = True
VERIFY_STRICT_VERSION = False

TOP_K = 12
MAX_ROWS = None              # e.g., 20 for quick test; None for full dataset
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "experiments", "pipeline_runs")  # you can change this


In [7]:
# --- Build pipeline ---
pipe = build_pipeline(
    framework_version=FRAMEWORK_VERSION,
    use_qur_live=USE_QUR_LIVE,
    use_org_profile=USE_ORG_PROFILE,
    verify_strict_extras=VERIFY_STRICT_EXTRAS,
    verify_strict_verbatim=VERIFY_STRICT_VERBATIM,
    verify_strict_version=VERIFY_STRICT_VERSION,
)
print("[OK] Pipeline initialized.")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[OK] Pipeline initialized.


In [8]:
# --- Load dataset and rewrites ---
gold_df = load_gold(FRAMEWORK_VERSION)
if MAX_ROWS is not None:
    gold_df = gold_df.head(int(MAX_ROWS))

qur_map = load_qur_map(FRAMEWORK_VERSION) if USE_QUR_PRECOMPUTED else {}
print("rows:", len(gold_df), "qur_map_ids:", len(qur_map))


rows: 100 qur_map_ids: 100


In [9]:
# --- Run batch ---
import pandas as pd
from tqdm.auto import tqdm

rows = []
for _, r in tqdm(gold_df.iterrows(), total=len(gold_df), desc=f"Pipeline batch ({FRAMEWORK_VERSION})"):
    gold_row = dict(r.to_dict())
    qid = str(gold_row.get("query_id", gold_row.get("id", "")))
    query = str(gold_row.get("original_query", gold_row.get("question", ""))).strip()

    rewrites = qur_map.get(qid, []) if USE_QUR_PRECOMPUTED else []

    out = pipe.answer(
        query,
        top_k=int(TOP_K),
        rewrites=rewrites if rewrites else None,
        gold_row=gold_row,
        use_generator=True,
        run_verify=True,
    )
    c = out.get("contract", out)

    rows.append({
        "query_id": qid,
        "framework_version": FRAMEWORK_VERSION,
        "question": query,
        "status": c.get("status"),
        "verifier_pass": bool(c.get("verifier_pass")),
        "verifier_errors": "|".join(list(c.get("verifier_errors") or [])),
        "primary_citation": c.get("primary_citation", ""),
        "all_citations": c.get("all_citations", ""),
        "odp_required_list": "|".join(list(c.get("odp_required_list") or [])),
        "selected_source_ids": "|".join(list(c.get("selected_source_ids") or [])),
        # Store full contract JSON for audit/debug
        "contract_json": json.dumps(c, ensure_ascii=False),
    })

df_out = pd.DataFrame(rows)
df_out.head()


Pipeline batch (rev5):   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,query_id,framework_version,question,status,verifier_pass,verifier_errors,primary_citation,all_citations,odp_required_list,selected_source_ids,contract_json
0,1,rev5,When must account managers and the designated ...,PARAMS_REQUIRED,False,ODPRequiredListNotEqualGold:gold=['ac-02_odp.0...,ac-2_smt.h,"ac-2_smt.h, ac-2_smt.h.1, ac-2_smt.h.2, ac-2_s...",ac-02_odp.02|ac-02_odp.05|ac-02_odp.06|ac-02_o...,ac-2_smt.h|ac-2_smt.h.1|ac-2_smt.h.2|ac-2_smt....,"{""answer_text"": ""Notify account managers and {..."
1,2,rev5,What must an organization enforce regarding lo...,OK,True,,ac-3_smt,"ac-3_smt, ac-3_gdn",,ac-3_smt|ac-3_gdn,"{""answer_text"": ""Enforce approved authorizatio..."
2,3,rev5,What principle should be employed to limit use...,OK,True,,ac-6_smt,"ac-6_smt, ac-6_gdn",,ac-6_smt|ac-6_gdn,"{""answer_text"": ""Employ the principle of least..."
3,4,rev5,How must a system be configured to handle cons...,PARAMS_REQUIRED,True,,ac-7_smt.a,"ac-7_smt.a, ac-7_smt.b, ac-7_gdn",ac-07_odp.01|ac-07_odp.02|ac-07_odp.03,ac-7_smt.a|ac-7_smt.b|ac-7_gdn,"{""answer_text"": ""Enforce a limit of {{ insert:..."
4,5,rev5,What actions must a system take to lock a sess...,PARAMS_REQUIRED,True,,ac-11_smt.b,"ac-11_smt.b, ac-11_smt.a",ac-11_odp.01,ac-11_smt.b|ac-11_smt.a,"{""answer_text"": ""Retain the device lock until ..."


In [10]:
# --- Save to Drive ---
import os, time
os.makedirs(OUTPUT_DIR, exist_ok=True)
ts = time.strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(OUTPUT_DIR, f"pipeline_{FRAMEWORK_VERSION}_contracts_{ts}.csv")
df_out.to_csv(csv_path, index=False)
print("[OK] Saved:", csv_path)

# Quick summary
print(df_out["status"].value_counts(dropna=False))
print("verifier_pass_rate:", float(df_out["verifier_pass"].mean()) if len(df_out) else 0.0)


[OK] Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/pipeline_runs/pipeline_rev5_contracts_20260219_044022.csv
status
PARAMS_REQUIRED    74
OK                 25
NO_EVIDENCE         1
Name: count, dtype: int64
verifier_pass_rate: 0.74
